# CUDA Optimization Course - Module 1: Profiling

Learn how to identify bottlenecks in your models using PyTorch profiler.

## Import Required Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.profiler import profile, record_function, ProfilerActivity
import numpy as np
import time
from typing import Dict, List
import sys

# Detect available device
if torch.cuda.is_available():
    device = 'cuda'
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
elif torch.backends.mps.is_available():
    device = 'mps'
    print("Using Apple Metal Performance Shaders (MPS)")
else:
    device = 'cpu'
    print("Using CPU")

print(f"Selected device: {device}")
print(f"PyTorch version: {torch.__version__}")

## Create Simple Model for Profiling

In [ ]:
class SimpleTransformer(nn.Module):
    def __init__(self, vocab_size=1000, d_model=256, nhead=4, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=1024,
                batch_first=True
            ),
            num_layers=num_layers
        )
        self.fc = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        x = self.fc(x)
        return x

model = SimpleTransformer().to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## Basic Profiling - Measure Execution Time

In [ ]:
batch_size = 32
seq_length = 128
input_ids = torch.randint(0, 1000, (batch_size, seq_length)).to(device)

# Warmup
with torch.no_grad():
    for _ in range(3):
        _ = model(input_ids)

# Measure forward pass time
torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
start = time.perf_counter()

with torch.no_grad():
    for _ in range(100):
        output = model(input_ids)

torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
forward_time = (time.perf_counter() - start) / 100

print(f"Forward pass time: {forward_time*1000:.2f} ms")
print(f"Throughput: {batch_size / forward_time:.0f} samples/sec")

## PyTorch Profiler - Detailed Analysis

In [ ]:
# Profile forward and backward pass
activities = [ProfilerActivity.CPU]
if device == 'cuda':
    activities.append(ProfilerActivity.CUDA)

with profile(
    activities=activities,
    record_shapes=True,
    profile_memory=True,
    with_stack=False,
    with_flops=True
) as prof:
    with record_function("forward_pass"):
        output = model(input_ids)
    
    with record_function("backward_pass"):
        loss = output.sum()
        loss.backward()

# Print top operations
print(prof.key_averages().table(sort_by="cuda_time_total" if device == 'cuda' else "cpu_time_total", row_limit=15))

## Memory Usage Analysis

In [ ]:
def print_memory_stats():
    if device == 'cuda':
        print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
        print(f"Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
        print(f"Max allocated: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
    elif device == 'mps':
        print("MPS doesn't provide memory stats via PyTorch API")
        print("Check Activity Monitor for GPU memory usage")
    else:
        print("CPU device - memory tracking limited")

print("Memory Statistics:")
print_memory_stats()

## Layer-wise Time Analysis

In [ ]:
class ProfilingTransformer(nn.Module):
    def __init__(self, vocab_size=1000, d_model=256, nhead=4, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=1024,
                batch_first=True
            )
            for _ in range(num_layers)
        ])
        self.fc = nn.Linear(d_model, vocab_size)
        self.layer_times = {}
    
    def forward(self, x):
        with record_function("embedding"):
            x = self.embedding(x)
        
        for i, layer in enumerate(self.layers):
            with record_function(f"transformer_layer_{i}"):
                x = layer(x)
        
        with record_function("output_projection"):
            x = self.fc(x)
        
        return x

profiling_model = ProfilingTransformer().to(device)

with profile(
    activities=activities,
    record_shapes=True
) as prof:
    output = profiling_model(input_ids)
    loss = output.sum()
    loss.backward()

print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=20))

## Key Takeaways

1. **Profiler Setup**: Use `torch.profiler.profile()` to measure execution time and memory
2. **Device Synchronization**: Always synchronize before measuring time
3. **Memory Tracking**: Monitor allocated vs reserved memory
4. **Layer Analysis**: Identify which layers consume most time
5. **Next Steps**: Use profiling results to guide optimization strategy